# 实验 2：AFT 删失回归 (AFT Censored Regression)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('.'))
%load_ext autoreload
%autoreload 2
import time
import numpy as np
import matplotlib.pyplot as plt
from models.aft import generate_aft_data
from algorithms.admm import run_u_admm, init_all_nodes
from algorithms.baselines import run_global_u_erm, run_dgd, run_d_proxgd
from utils.excel_utils import append_to_excel
from utils.eval_utils import calculate_metrics, evaluate_correlation

# === 1. 定义评估辅助函数 ===
def print_full_metrics(name, theta_est, theta_true, d_aft):
    m1 = calculate_metrics(theta_true, theta_est)
    corr = evaluate_correlation(d_aft['X'], theta_true, theta_est)
    pair = (corr['Kendall_Corr'] + 1) / 2
    print(f"{name:<20} | "
          f"RMSE: {m1['RMSE']:.4f} | "
          f"F1: {m1['F1_Score']:.4f} | "
          f"Prec: {m1['Precision']:.4f} | "
          f"Rec: {m1['Recall']:.4f} | "
          f"Ken: {corr['Kendall_Corr']:.4f} | "
          f"Pear: {corr['Pearson_Corr']:.4f} | "
          f"Pair: {pair:.4f}")

# -- 参数 --
params = {
    'Experiment': 'AFT Survival',
    'm': 10, 
    'n': 200,
    'p_prime': 5, 
    'p': 20, 
    'pc': 0.3,
    'T': 40, 
    'W_inner': 5, 
    'rho': 1.3, 
    'ic_type': 'bic', 
    'lambda_candidates': np.logspace(-2.5, -1.5, 10).tolist(),
    # 'lambda_dgd': np.logspace(-2.5, -1.5, 10).tolist(),
    # 'lambda_d_proxgd': np.logspace(-2.5, -1.5, 10).tolist(),
    'noise_type': 't1',
    'rng_seed': 245,
    'dgd_lr': 0.1,
    'd_proxgd_lr': 0.1,
    'run_Global': True,
    'run_DGD': True,
    'run_D_ProxGD': True,
}

m = params['m']
n = params['n']
p = params['p']
noise_type = params['noise_type']
W_inner = params['W_inner']
T = params['T']
total_steps = T * W_inner

np.random.seed(params['rng_seed'])

# 2. 生成数据
d_aft = generate_aft_data(
    m=params['m'], n=params['n'], p=params['p'], p_prime=params['p_prime'], 
    pc=params['pc'], noise_type=params['noise_type'], rng_seed=params['rng_seed']
)
theta_true = d_aft['theta_true']

# 热启动初始化
theta0_list, theta_naive = init_all_nodes(d_aft)
d_aft['theta0_list'] = theta0_list
d_aft['theta_naive'] = theta_naive

# A. U-ADMM (Proposed)
t0 = time.time()
theta_u_a, theta_n_a, hist_a = run_u_admm(
    d_aft, T=T, W_inner=W_inner, 
    rho=params['rho'], verbose=True,
    lambda_candidates=params['lambda_candidates'],
    ic_type=params.get('ic_type', 'bic'),
    theta0_list=theta0_list
)
theta_uadmm = theta_u_a[0]
theta_avg = theta_n_a
metrics_avg = calculate_metrics(theta_true, theta_avg)
x_uadmm = np.arange(len(hist_a['rmse'])) * W_inner
print(f'U-ADMM 耗时: {time.time() - t0:.1f}s')

# Local
local_metrics = [calculate_metrics(theta_true, th) for th in theta0_list]
local_corrs = [evaluate_correlation(d_aft['X'], theta_true, th) for th in theta0_list]
rmse_local = np.mean([lm['RMSE'] for lm in local_metrics])
f1_local = np.mean([lm['F1_Score'] for lm in local_metrics])
prec_local = np.mean([lm['Precision'] for lm in local_metrics])
rec_local = np.mean([lm['Recall'] for lm in local_metrics])
ken_local = np.mean([c['Kendall_Corr'] for c in local_corrs])

# B. Global U-ERM
if params.get('run_Global', True):
    t0 = time.time()
    theta_global, hist_global = run_global_u_erm(d_aft, n_iter=total_steps, lambda_candidates=params['lambda_candidates'], ic_type=params.get('ic_type', 'bic'), init_theta=theta_naive, return_history=True)
    rmse_global = calculate_metrics(theta_true, theta_global)['RMSE']
    print(f'Global 耗时: {time.time()-t0:.1f}s')

# C. D-subGD
if params.get('run_DGD', True):
    t0 = time.time()
    theta_dgd, hist_dgd = run_dgd(d_aft, T=total_steps, lr=params.get('dgd_lr', 0.1), lambda_candidates=params.get('lambda_candidates'), ic_type=params.get('ic_type', 'bic'), theta_init_list=theta0_list, return_history=True)
    rmse_dgd = calculate_metrics(theta_true, theta_dgd)['RMSE']
    print(f'D-subGD 耗时: {time.time()-t0:.1f}s')

# D. D-ProxGD
if params.get('run_D_ProxGD', True):
    t0 = time.time()
    theta_d_proxgd, hist_d_proxgd = run_d_proxgd(d_aft, T=total_steps, lr=params.get('d_proxgd_lr', 0.1), lambda_candidates=params.get('lambda_candidates'), ic_type=params.get('ic_type', 'bic'), theta_init_list=theta0_list, return_history=True)
    rmse_d_proxgd = calculate_metrics(theta_true, theta_d_proxgd)['RMSE']
    print(f'D-ProxGD 耗时: {time.time()-t0:.1f}s')

# === 对比表 ===
print(f'\n{"Algorithm":<20} | {"RMSE":<7} | {"F1":<7} | {"Prec":<7} | {"Rec":<7} | {"Kendall":<7} | {"Pearson":<7} | {"Pairwise":<7}')
print("-" * 110)
print_full_metrics('U-ADMM', theta_uadmm, theta_true, d_aft)

pear_local = np.mean([c['Pearson_Corr'] for c in local_corrs])
pair_local = (ken_local + 1) / 2
print(f"{'Local':<20} | RMSE: {rmse_local:.4f} | F1: {f1_local:.4f} | Prec: {prec_local:.4f} | Rec: {rec_local:.4f} | Ken: {ken_local:.4f} | Pear: {pear_local:.4f} | Pair: {pair_local:.4f}")

if 'theta_global' in locals() and params.get('run_Global', True):
    print_full_metrics('Global', theta_global, theta_true, d_aft)

if 'theta_dgd' in locals() and params.get('run_DGD', True):
    print_full_metrics('D-subGD', theta_dgd, theta_true, d_aft)

if 'theta_d_proxgd' in locals() and params.get('run_D_ProxGD', True):
    print_full_metrics('D-ProxGD', theta_d_proxgd, theta_true, d_aft)

print_full_metrics('Avg', theta_avg, theta_true, d_aft)


In [ ]:
# === 3. RMSE 收敛对比图 ===
fig, ax = plt.subplots(figsize=(9, 5.5))

style_map = {
    'U-ADMM': ('-o', '#3498DB'), 
    'Global': ('-', '#FFA500'),
    'D-subGD': ('--', '#E67E22'),
    'D-ProxGD': ('-', '#9B59B6')
}

# 寻找最大步数
max_steps = params.get('T', 40) * params.get('W_inner', 5)

# 1. U-ADMM
x_uadmm = np.arange(len(hist_a['rmse'] if 'aft' == 'aft' else hist_r['rmse'])) * params.get('W_inner', 5)
hist_uadmm = hist_a['rmse'] if 'aft' == 'aft' else hist_r['rmse']
ax.plot(x_uadmm, hist_uadmm, marker=style_map['U-ADMM'][0][1], markersize=4, lw=2,
        label=f"U-ADMM (RMSE={hist_uadmm[-1]:.4f})", color=style_map['U-ADMM'][1], zorder=5)

# 2. Avg
ax.hlines(metrics_avg['RMSE'], xmin=0, xmax=max_steps, color='#E74C3C', linestyle='--', lw=1.8,
          label=f"Avg (RMSE={metrics_avg['RMSE']:.4f})")

# 3. Local
if 'rmse_local' in locals():
    ax.hlines(rmse_local, xmin=0, xmax=max_steps, color='#2ECC71', linestyle='-.', lw=1.8,
              label=f"Local (RMSE={rmse_local:.4f})")

# 4. Global
if 'theta_global' in locals() and params.get('run_Global', True):
    if 'hist_global' in locals() and hist_global is not None and 'rmse' in hist_global:
        x_g = np.arange(len(hist_global['rmse']))
        ax.plot(x_g, hist_global['rmse'], color=style_map['Global'][1], linestyle=style_map['Global'][0], lw=1.8,
                label=f"Global (RMSE={rmse_global:.4f})")
    else:
        ax.hlines(rmse_global, xmin=0, xmax=max_steps, color=style_map['Global'][1], linestyle=style_map['Global'][0], lw=1.8,
                  label=f"Global (RMSE={rmse_global:.4f})")

# 5. D-subGD
if 'theta_dgd' in locals() and params.get('run_DGD', True):
    if 'hist_dgd' in locals() and hist_dgd is not None and 'rmse' in hist_dgd:
        x_d = np.arange(len(hist_dgd['rmse']))
        ax.plot(x_d, hist_dgd['rmse'], color=style_map['D-subGD'][1], linestyle=style_map['D-subGD'][0], lw=1.8,
                label=f"D-subGD (RMSE={rmse_dgd:.4f})")
    else:
        ax.hlines(rmse_dgd, xmin=0, xmax=max_steps, color=style_map['D-subGD'][1], linestyle=style_map['D-subGD'][0], lw=1.8,
                  label=f"D-subGD (RMSE={rmse_dgd:.4f})")

# 6. D-ProxGD
if 'theta_d_proxgd' in locals() and params.get('run_D_ProxGD', True):
    if 'hist_d_proxgd' in locals() and hist_d_proxgd is not None and 'rmse' in hist_d_proxgd:
        x_d_proxgd = np.arange(len(hist_d_proxgd['rmse']))
        ax.plot(x_d_proxgd, hist_d_proxgd['rmse'], color=style_map['D-ProxGD'][1], linestyle=style_map['D-ProxGD'][0], lw=1.8,
                label=f"D-ProxGD (RMSE={rmse_d_proxgd:.4f})")
    else:
        ax.hlines(rmse_d_proxgd, xmin=0, xmax=max_steps, color=style_map['D-ProxGD'][1], linestyle=style_map['D-ProxGD'][0], lw=1.8,
                  label=f"D-ProxGD (RMSE={rmse_d_proxgd:.4f})")

ax.set_xlabel(f'Total gradient / consensus steps (W_inner={W_inner})', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title(f'RMSE Convergence: {noise_type} Noise (m={m}, n={n})', fontsize=14)
ax.legend(fontsize=10, bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.35)
plt.tight_layout()

os.makedirs('aft', exist_ok=True)
plt.savefig('aft/convergence_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'图已保存到 aft/convergence_comparison.png')


In [ ]:
# 一致性变化图 (方差图)
debug_history = hist_a['debug']
T_iters = len(debug_history)
variances = np.zeros((T_iters, p))

for t in range(T_iters):
    theta_t = debug_history[t]['theta_t']
    theta_mat = np.hstack(theta_t)
    variances[t, :] = np.var(theta_mat, axis=1)

plt.figure(figsize=(10, 6))
for i in range(params['p_prime']):
    plt.plot(range(T_iters), variances[:, i], label=f'Dim {i+1} (True=1)')
for i in range(params['p_prime'], min(params['p_prime']+3, p)):
    plt.plot(range(T_iters), variances[:, i], linestyle='--', label=f'Dim {i+1} (True=0)')

plt.title('Variance of Coefficients Across Nodes Over Iterations')
plt.xlabel('Outer Iteration')
plt.ylabel('Variance')
plt.yscale('log')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# -- 系数对比图 --
plt.figure(figsize=(10, 5))
# 1. True
plt.plot(theta_true, marker='o', markersize=6, linestyle='None', label='True Coefficients', alpha=0.5, color='gray')
# 2. U-ADMM
plt.plot(theta_uadmm, marker='x', markersize=8, color='#4E93D9', linestyle='None', alpha=1, label='U-ADMM')
# 3. Global
if 'theta_global' in locals() and params.get('run_Global', True):
    plt.plot(theta_global, marker='+', markersize=8, color='green', linestyle='None', label='Global')
# 4. D-subGD
if 'theta_dgd' in locals() and params.get('run_DGD', True):
    plt.plot(theta_dgd, marker='x', markersize=8, color='#EE1C25', linestyle='None', label='D-subGD')
# 5. D-ProxGD
if 'theta_d_proxgd' in locals() and params.get('run_D_ProxGD', True):
    plt.plot(theta_d_proxgd, marker='D', markersize=6, color='#9B59B6', linestyle='None', alpha=0.9, label='D-ProxGD')

plt.axhline(0, color='black', alpha=0.2, linestyle='--')
plt.xlabel('Coefficient Index')
plt.ylabel('Coefficient Value')
plt.title('Coefficient Comparison: True vs All Methods')
plt.legend(frameon=True, loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('aft', exist_ok=True)
plt.savefig('aft/coefficient_comparison.png', dpi=150)
plt.show()


In [ ]:
# -- 保存到 Excel --
import time

result_dict = {
    'Time': time.strftime('%Y-%m-%d %H:%M:%S'),
    'm': m, 'n': params['n'], 'p': p, 'T': T, 'W_inner': W_inner,
    'Noise': params['noise_type'],
    'RMSE_Proposed': calculate_metrics(theta_true, theta_uadmm)['RMSE'],
    'F1_Proposed': calculate_metrics(theta_true, theta_uadmm)['F1_Score'],
    'Kendall_Proposed': evaluate_correlation(d_aft['X'], theta_true, theta_uadmm)['Kendall_Corr'],
}

if 'rmse_local' in locals():
    result_dict['RMSE_Local'] = rmse_local
if 'rmse_global' in locals():
    result_dict['RMSE_Global'] = rmse_global
if 'rmse_dgd' in locals():
    result_dict['RMSE_DsubGD'] = rmse_dgd
if 'rmse_d_proxgd' in locals():
    result_dict['RMSE_D-ProxGD'] = rmse_d_proxgd

# 记录前 p 维度的系数对比
for i in range(p):
    result_dict[f'theta_{i}_true'] = float(np.squeeze(theta_true[i]))
    result_dict[f'theta_{i}_uadmm'] = float(np.squeeze(theta_uadmm[i]))
    if 'theta_global' in locals():
        result_dict[f'theta_{i}_global'] = float(np.squeeze(theta_global[i]))
    if 'theta_dgd' in locals():
        result_dict[f'theta_{i}_dgd'] = float(np.squeeze(theta_dgd[i]))
    if 'theta_d_proxgd' in locals():
        result_dict[f'theta_{i}_d_proxgd'] = float(np.squeeze(theta_d_proxgd[i]))

excel_path = 'aft/results.xlsx'
os.makedirs('aft', exist_ok=True)
append_to_excel(excel_path, result_dict)
print(f'实验结果已保存到 {excel_path}')
